# AeroFleet — 1,000-case mass forensics run on Colab (free GPU/CPU)

Colab-side counterpart to `research/kaggle_mass_forensics_run.ipynb`. Same harness, same real
Ollama backend, same non-blocking background-process pattern so you can check progress without
stopping the run. Runs a **different, non-overlapping batch range** so it works in parallel with
the Kaggle run (and a third machine, if you have one).

## Required one-time manual setup (Colab UI, not this notebook)

1. **Runtime → Change runtime type.** GPU is optional now — the model (3.8B, ~3.2GB) runs fine
   on CPU alone. Pick T4 GPU if you have quota to spare, or leave it on CPU if not; either works.
2. **Secrets** (key icon in the left sidebar) → **Add new secret** → name it `GH_PAT`, value =
   a GitHub fine-grained token, read-only, scoped to just `AdityaPathare46/aerofleet` (private
   repo — reuse the same token you made for Kaggle). Toggle **Notebook access** on.
3. This notebook mounts your Google Drive and writes results there directly — progress survives a
   disconnect automatically, no manual save step needed. First time only: **File → Save a copy in
   Drive** so you can reopen this notebook tomorrow without re-uploading it.

## Which batch range this notebook runs

`--target 1000 --batch-size 25` gives 40 batches total.

| Machine | Batches | Cases |
|---|---|---|
| Kaggle | 1-14 | MFI-00001 – MFI-00350 |
| **This Colab notebook** | **27-40** | **MFI-00651 – MFI-01000** |
| Third machine | 15-26 | MFI-00351 – MFI-00650 |

Change `ONLY_BATCHES` in the run cell if you want a different split — just keep all machines'
ranges disjoint and covering 1-40 together.

## Model roster

Every agent shares one model — `phi4-mini-reasoning` (Microsoft, 3.8B, ~3.2GB, verified real and
pullable) — chosen to run on ordinary hardware without the VRAM-thrashing a larger, multi-model
roster caused on constrained GPUs. **Every machine in the split above runs the identical real
roster** — no per-machine model substitution, no methodology deviation to track.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU visible — that's fine, phi4-mini-reasoning runs on CPU."


## 1. Mount Drive (for persistence) and clone the private repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from google.colab import userdata
_token = userdata.get('GH_PAT')

REPO_DIR = "/content/aerofleet"
import os
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://{_token}@github.com/AdityaPathare46/aerofleet.git {REPO_DIR}
else:
    print("Repo already present — pulling latest instead of a fresh clone.")
    !cd {REPO_DIR} && git pull

del _token
%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 2. Install Ollama and start the server in this session

`OLLAMA_KEEP_ALIVE=-1` tells Ollama to never voluntarily unload the model due to idle time —
defensive, though with a single ~3.2GB model there's no other model competing for space so this
mostly just avoids an unnecessary reload if calls are spaced apart.


In [ ]:
# zstd: required by the Ollama installer to extract its archive.
# pciutils (lspci): lets the installer auto-detect the GPU and install the
# matching CUDA runtime — without it, install silently warns and may fall
# back to a CPU-only build, which would make the run far too slow to finish.
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time, requests, os

os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

log = open("/content/ollama_serve.log", "a")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=5)
        print("Ollama server is up.")
        break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not come up — check /content/ollama_serve.log")

time.sleep(2)
!grep -i -E "gpu|cuda|library=" /content/ollama_serve.log | tail -5


## 3. Pull the model

One model now (~3.2GB), not four. Re-run this cell if interrupted — Ollama resumes partial
downloads.


In [ ]:
print("--- pulling phi4-mini-reasoning ---")
!ollama pull phi4-mini-reasoning
!ollama list


## 4. Study directory on Drive (persists automatically) + this machine's batch range

No resume step needed like Kaggle — since `STUDY_DIR` lives on Drive, a previous session's
`results.jsonl` is just already there.


In [ ]:
from pathlib import Path

STUDY_DIR = Path("/content/drive/MyDrive/aerofleet_mass_forensics_colab")
STUDY_DIR.mkdir(parents=True, exist_ok=True)

ONLY_BATCHES = "27-40"  # this machine's assigned, non-overlapping slice — see the table above.

n_prior = sum(1 for _ in open(STUDY_DIR / "results.jsonl")) if (STUDY_DIR / "results.jsonl").exists() else 0
print(f"{n_prior} case-attempts already on Drive from a previous session (0 is normal for a first run).")


## 5. Record this run's configuration (must match the other machines)

No per-agent model overrides needed — every agent already shares the same `phi4-mini-reasoning`
model via `AgentFactory.DEFAULT_MODEL_MAP`. This cell just records the run's metadata.


In [ ]:
import os, json, subprocess, datetime

os.environ["OLLAMA_HOST"] = "http://localhost:11434"
os.environ.pop("USE_MOCK_AGENTS", None)

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip() or "CPU (no GPU visible)"

metadata = {
    "platform": "colab",
    "gpu": gpu_name,
    "only_batches": ONLY_BATCHES,
    "run_started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_roster": {
        "shared_model": "phi4-mini-reasoning",
        "note": "every agent uses this same model — no per-machine substitution needed",
    },
}
with open(STUDY_DIR / "cloud_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


## 6. Start the harness — runs in the background

This does NOT block the notebook — it starts the run and returns immediately. Use the cells below
to check on it, any time, as often as you like, without needing to stop it first.


In [ ]:
import subprocess, os, requests

# Confirm Ollama is actually reachable BEFORE starting the harness.
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama is reachable:", r.json())
except requests.exceptions.RequestException as e:
    raise RuntimeError(
        f"Ollama is NOT reachable ({e}). Do not start the harness yet — go back and re-run "
        "the 'start server' cell first, then re-run this cell."
    )

harness_log_path = STUDY_DIR / "harness_run.log"
harness_log = open(harness_log_path, "a")

cmd = [
    "python", "-m", "scenario_engine.mass_forensics_evaluation",
    "--study-dir", str(STUDY_DIR), "--target", "1000", "--batch-size", "25",
    "--retry-failed-max", "5",
]
if ONLY_BATCHES:
    cmd += ["--only-batches", ONLY_BATCHES]

harness_proc = subprocess.Popen(cmd, stdout=harness_log, stderr=subprocess.STDOUT, env=os.environ.copy())
print(f"Harness started in the background, PID {harness_proc.pid}.")
print(f"Logging to {harness_log_path}")
print("Run the cells below any time to check progress — no need to wait or stop this first.")


## 7. Monitoring — run any of these any time, as often as you like


In [ ]:
# Tail the live log
!tail -n 40 {harness_log_path}


In [ ]:
# Is it still running? (None = still running; a number = exit code, it finished/stopped)
print("exit code (None = still running):", harness_proc.poll())


In [ ]:
# How many case-attempts recorded so far
results_file = STUDY_DIR / "results.jsonl"
if results_file.exists():
    with open(results_file) as f:
        n = sum(1 for _ in f)
    print(f"{n} case-attempts recorded so far in this machine's range (27-40).")
else:
    print("No results yet.")


In [ ]:
# Thrashing check: if this shows a model, then a few cells later shows a DIFFERENT model
# (or nothing) repeatedly, models are being evicted and reloaded between calls — worth flagging.
!ollama ps


## 8. Stopping cleanly

Use this instead of Colab's "interrupt execution" — it terminates only the harness process,
leaving the Ollama server running undisturbed (an interrupt at the notebook level may not make
that distinction).


In [ ]:
harness_proc.terminate()
harness_proc.wait(timeout=30)
print("Harness stopped cleanly. results.jsonl on Drive has everything completed up to this point.")


## 9. Resuming tomorrow

Reopen this notebook from Drive, Runtime → reconnect, reselect T4 GPU, and re-run every cell from
the top (the Colab machine itself is wiped — repo clone, Ollama install, and model pull all
happen again). No manual resume step needed: since `STUDY_DIR` already points at the same Drive
folder, the harness sees yesterday's `results.jsonl` there automatically and just continues.

## 10. Merging all three machines' results

Once Kaggle, this Colab notebook, and the third machine have each made progress (they don't all
need to be *finished* — merging is safe at any point), pull all three `results.jsonl` +
`dataset_manifest.json` + `run_config.json` sets down to one place and run:

```bash
python -m research.merge_batched_results \
    --source /path/to/kaggle_study \
    --source /path/to/colab_study \
    --source /path/to/friend_study \
    --output /path/to/merged_study

python -m scenario_engine.mass_forensics_evaluation --study-dir /path/to/merged_study --report-only
```

The merge script refuses to combine studies that weren't generated with the same
`--target`/`--batch-size`/`--seed` (they wouldn't share a manifest), and warns if the same
case_id shows up from two sources — a sign the batch ranges weren't actually disjoint.
